In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Crypto_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/bitcoin_1m.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20)

df.printSchema()

Raw Dataset Loaded
Total Rows: 1000
+-------------------+--------+--------+--------+--------+-----------+
|          timestamp|    open|    high|     low|   close|     volume|
+-------------------+--------+--------+--------+--------+-----------+
|2026-06-10 16:45:00|61213.22|61224.46|61183.66|61203.23|22.44644591|
|2026-06-10 16:46:00|61203.18|61203.19| 61150.0|61151.14| 1.22304211|
|2026-06-10 16:47:00|61137.01|61137.01|61115.62|61115.62| 1.53656201|
|2026-06-10 16:48:00|61114.78|61141.42|61109.45|61141.42| 0.28438097|
|2026-06-10 16:49:00|61127.01|61128.98|61102.05|61128.98| 0.34435015|
|2026-06-10 16:50:00| 61131.1|61184.23|61128.72|61184.23|    0.45573|
|2026-06-10 16:51:00|61196.96|61237.42|61189.76|61232.47| 0.45532274|
|2026-06-10 16:52:00|61232.49| 61276.0|61220.42| 61223.0| 0.22844085|
|2026-06-10 16:53:00|61220.38|61220.38|61183.77|61183.78| 0.03991363|
|2026-06-10 16:54:00|61203.71|61267.76|61199.99|61260.35| 0.16489444|
|2026-06-10 16:55:00|61257.65|61294.01|61257.65|61294.

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+---------+----+----+---+-----+------+
|timestamp|open|high|low|close|volume|
+---------+----+----+---+-----+------+
|        0|   0|   0|  0|    0|     0|
+---------+----+----+---+-----+------+

Total Rows: 1000
Unique Rows: 1000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn("timestamp", F.to_timestamp("timestamp"))
df = df.orderBy("timestamp")

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2026-06-10 16:45:00|2026-06-11 09:24:00|
+-------------------+-------------------+



In [6]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("prev_time", F.lag("timestamp").over(w))

df = df.withColumn(
    "diff_min",
    (F.unix_timestamp("timestamp")
     - F.unix_timestamp("prev_time")) / 60
)

df.select(
    "timestamp",
    "prev_time",
    "diff_min"
).show(20, False)

gap_count = df.filter(F.col("diff_min") > 1).count()

print("Gap Count:", gap_count)

+-------------------+-------------------+--------+
|timestamp          |prev_time          |diff_min|
+-------------------+-------------------+--------+
|2026-06-10 16:45:00|NULL               |NULL    |
|2026-06-10 16:46:00|2026-06-10 16:45:00|1.0     |
|2026-06-10 16:47:00|2026-06-10 16:46:00|1.0     |
|2026-06-10 16:48:00|2026-06-10 16:47:00|1.0     |
|2026-06-10 16:49:00|2026-06-10 16:48:00|1.0     |
|2026-06-10 16:50:00|2026-06-10 16:49:00|1.0     |
|2026-06-10 16:51:00|2026-06-10 16:50:00|1.0     |
|2026-06-10 16:52:00|2026-06-10 16:51:00|1.0     |
|2026-06-10 16:53:00|2026-06-10 16:52:00|1.0     |
|2026-06-10 16:54:00|2026-06-10 16:53:00|1.0     |
|2026-06-10 16:55:00|2026-06-10 16:54:00|1.0     |
|2026-06-10 16:56:00|2026-06-10 16:55:00|1.0     |
|2026-06-10 16:57:00|2026-06-10 16:56:00|1.0     |
|2026-06-10 16:58:00|2026-06-10 16:57:00|1.0     |
|2026-06-10 16:59:00|2026-06-10 16:58:00|1.0     |
|2026-06-10 17:00:00|2026-06-10 16:59:00|1.0     |
|2026-06-10 17:01:00|2026-06-10